<a href="https://colab.research.google.com/github/kumarsirish/FDP-AGENENTIC-AI-RAG/blob/main/rag-agentic-01/fictional-depatment-rag-agentic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple Agentic RAG

This notebook walks through building an Agentic RAG system using LangChain and FAISS.

**Steps:**
1. Install dependencies
2. Ingest documents into a FAISS vector store
3. Build a LangChain agent with a document search tool
4. Ask questions

## 1. Install Dependencies

In [1]:
import os

if not os.path.exists('requirements.txt'):
    !wget https://raw.githubusercontent.com/kumarsirish/FDP-AGENENTIC-AI-RAG/main/rag-agentic-01/requirements.txt
else:
    print('requirements.txt already exists.')
%pip install -r  requirements.txt -q


requirements.txt already exists.


## 2. Set API Keys

In [2]:
import os
from dotenv import load_dotenv

# Fallback: try Google Colab secrets
try:
    from google.colab import userdata
    if not os.getenv("HF_TOKEN"):
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    if not os.getenv("GEMINI_API_KEY"):
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY") or ""
except ImportError:
    # Load from .env file (local development)
    load_dotenv("/home/sirkumar/FDP-AGENENTIC-AI-RAG/.env")

HF_TOKEN = (os.getenv("HF_TOKEN") or "").strip()
GEMINI_API_KEY = (os.getenv("GEMINI_API_KEY") or "").strip()


if not GEMINI_API_KEY or not HF_TOKEN:
    print("Warning: HF_TOKEN or GEMINI_API_KEY not set. Please set it in .env file or Colab secrets.")

In [3]:
department_info = [
    # Department overview
    "The Department of Quantum Engineering (DQE) has around 140 students and 20 professors, focusing on applied quantum computing and intelligent systems.",
    "Students can choose from 5 courses ranging from core subjects to electives and hands-on project work, with strong industry exposure.",
    "DQE offers a postgraduate module 'Foundations of Quantum AI' and an undergraduate elective 'Quantum Machine Learning' using IBM Qiskit and PennyLane.",
    "All students have access to cloud quantum hardware through IBM Quantum Network and Amazon Braket as part of their coursework.",

    # Quantum AI research
    "DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence.",
    "The Quantum AI Lab investigates Quantum Neural Networks (QNNs) using parameterized quantum circuits (PQCs) and collaborates with a national quantum computing centre on 20-qubit and 50-qubit processors.",
    "A key research thread is Quantum Reinforcement Learning (QRL), where quantum agents learn policies faster than classical counterparts on specific problem classes.",
    "Project QuLearn is DQE's flagship initiative building hybrid classical-quantum models for drug discovery and materials science.",

]


## 3. Ingest Documents into FAISS Vector Store

Defines knowledge inline, splits it into chunks, embeds them, and saves the vector store locally.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

documents = [Document(page_content=text) for text in department_info]

splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=50)
docs = splitter.split_documents(documents)

# Force anonymous download for public embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L12-v2"
)

db = FAISS.from_documents(docs, embeddings)
db.save_local("vectorstore")

print(f"Vector DB created successfully ({len(docs)} chunks)")

/tmp/ipykernel_14149/1652518417.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector DB created successfully (8 chunks)


In [5]:
# Create retriever — returns top 2 most similar documents for any query
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 2})
print("Retriever ready.")

Retriever ready.


## 4. Build the RAG Agent

Loads the vector store and wires it up as a tool for a LangChain ZERO_SHOT_REACT agent.

In [6]:
from langchain_core.tools import tool
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model


# ── 1. Load vector store ──────────────────────────────────────────────

db = FAISS.load_local(
    "vectorstore",
    embeddings,
    allow_dangerous_deserialization=True
)
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 2})

# ── 2. Tool ───────────────────────────────────────────────────────────
@tool
def search_answers(query: str) -> str:
    """Search the knowledge base for information about the DQE department."""
    results = retriever.invoke(query)
    docs = "\n".join([doc.page_content for doc in results])
    print(f"\n[TOOL CALLED] query='{query}'")
    print(f"[RETRIEVED]:\n{docs}\n")
    return docs if docs.strip() else "No relevant documents found."



llm_endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    provider="auto",           # ← KEY FIX: routes to Nebius/Fireworks/etc.
    huggingfacehub_api_token=HF_TOKEN,
    task="text-generation",
    max_new_tokens=512,
    temperature=0.1,
)


# Wrap in ChatHuggingFace — this enables bind_tools() for agentic RAG
chat_llm = ChatHuggingFace(llm=llm_endpoint, verbose=True)
#chat_llm = init_chat_model("google_genai:gemini-2.5-flash", api_key=GEMINI_API_KEY)

#

# ── 4. Agent ──────────────────────────────────────────────────────────
agent = create_agent(
    model=chat_llm,
    tools=[search_answers],
    system_prompt=(
        "You are a helpful assistant for the DQE department. "
        "ALWAYS call the 'search_answers' tool before answering. "
        "Only use the retrieved documents to form your answer."
    ),
)

print(f"Agent ready ✅")

Agent ready ✅


## 5. Ask Questions

Edit `question` below and re-run this cell to query the agent.

In [7]:
def ask(question: str) -> str:
    response = agent.invoke({"messages": [{"role": "user", "content": question}]})
    # Get the last message (final answer from agent)
    return response["messages"][-1].content

print("Answer: ",ask("What is DQE?"))




[TOOL CALLED] query='What is DQE?'
[RETRIEVED]:
DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence.
The Department of Quantum Engineering (DQE) has around 140 students and 20 professors, focusing on applied quantum computing and intelligent systems.

Answer:  DQE, or the Department of Quantum Engineering, primarily focuses on the research area of Quantum AI, which is the intersection of quantum computing and artificial intelligence. The department has approximately 140 students and 20 professors, with a specific emphasis on applied quantum computing and intelligent systems.


In [8]:
print("Answer: ",ask("How many students does DQE have?"))


[TOOL CALLED] query='How many students does DQE have?'
[RETRIEVED]:
The Department of Quantum Engineering (DQE) has around 140 students and 20 professors, focusing on applied quantum computing and intelligent systems.
All students have access to cloud quantum hardware through IBM Quantum Network and Amazon Braket as part of their coursework.

Answer:  The Department of Quantum Engineering (DQE) has around 140 students.


In [9]:
print("Answer: ",ask("Whats the objective of DQE department"))


[TOOL CALLED] query='objective of DQE department'
[RETRIEVED]:
DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence.
The Department of Quantum Engineering (DQE) has around 140 students and 20 professors, focusing on applied quantum computing and intelligent systems.

Answer:  The primary objective of the DQE department appears to be focused on Quantum AI, specifically the intersection of quantum computing and artificial intelligence. The department also emphasizes applied quantum computing and intelligent systems, with approximately 140 students and 20 professors dedicated to these areas of research.
